# Fine-tuning Qwen2.5-7B-Instruct for Named Entity Recognition

Run on **Google Colab** with a GPU runtime (T4 or A100 recommended).

## Experiment Grid

| Category | Configs |
|---|---|
| Baselines | Zero-shot, Few-shot (5 in-context examples) |
| Generic only | 0.5k, 1k, 2k, 4k samples |
| Domain (tech) only | 0.5k, 1k, 2k, 4k samples |
| Generic → Domain | 0.5k+0.5k, 1k+1k, 2k+2k, 4k+4k (two-phase sequential SFT) |

**Dataset:** [NuNER](https://huggingface.co/datasets/numind/NuNER) — preprocessed by `preprocessing.ipynb`
**Metric:** Entity-level micro F1 (exact match on entity text + type, case-insensitive)


In [ ]:
# ── Step 0: check GPU ────────────────────────────────────────────────────────
!nvidia-smi


In [ ]:
# ── Step 1: install dependencies ─────────────────────────────────────────────
# Unsloth recommends checking https://github.com/unslothai/unsloth for the
# latest Colab install command. The line below works for most recent Colab runtimes.
!pip install unsloth -q
!pip install "trl>=0.9.0" "accelerate>=0.35.0" bitsandbytes -q
print("Installation complete.")


In [ ]:
import ast, gc, json, re, warnings
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# ── Configuration — edit as needed ───────────────────────────────────────────
MODEL_NAME    = "unsloth/Qwen2.5-7B-Instruct"
DATA_DIR      = Path("data")          # created by preprocessing.ipynb
RESULTS_CSV   = Path("results_qwen25_7b.csv")

# LoRA
MAX_SEQ_LEN   = 2048
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05

# Training
TRAIN_EPOCHS  = 1           # epochs per training phase
LEARNING_RATE = 2e-4
BATCH_SIZE    = 2
GRAD_ACCUM    = 4           # effective batch size = BATCH_SIZE * GRAD_ACCUM = 8

# Evaluation
EVAL_SAMPLE   = 500         # max test samples; set None to evaluate on full test set
FEW_SHOT_N    = 5           # in-context examples for the few-shot baseline
SEED          = 42


In [ ]:
# ── Data utilities ────────────────────────────────────────────────────────────

def parse_entity_set(output_str: str) -> set:
    """Parse a NuNER output string into a set of (entity_text, entity_type) tuples."""
    match = re.search(r'\[.*?\]', str(output_str), re.DOTALL)
    if not match:
        return set()
    try:
        items = ast.literal_eval(match.group())
    except (ValueError, SyntaxError):
        return set()
    result = set()
    for item in items:
        if isinstance(item, str) and ' <> ' in item:
            parts = item.split(' <> ', maxsplit=1)
            if len(parts) == 2:
                result.add((parts[0].strip().lower(), parts[1].strip().lower()))
    return result


def entity_f1(predictions: list, ground_truths: list) -> dict:
    """Compute entity-level micro Precision / Recall / F1."""
    tp = fp = fn = 0
    for pred_str, gt_str in zip(predictions, ground_truths):
        pred_set = parse_entity_set(pred_str)
        gt_set   = parse_entity_set(gt_str)
        tp += len(pred_set & gt_set)
        fp += len(pred_set - gt_set)
        fn += len(gt_set  - pred_set)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {
        "precision": round(precision, 4),
        "recall":    round(recall,    4),
        "f1":        round(f1,        4),
        "tp": tp, "fp": fp, "fn": fn,
    }


def format_chat(input_text: str, output_text: str, tokenizer) -> str:
    """Format a NuNER sample as a chat turn for SFT (uses the model's chat template)."""
    messages = [
        {"role": "user",      "content": input_text},
        {"role": "assistant", "content": output_text},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


def generate_response(
    model, tokenizer, input_text: str,
    few_shot_examples: list | None = None,
    max_new_tokens: int = 256,
) -> str:
    """Run inference and return the raw text output (stripped)."""
    messages = []
    for ex in (few_shot_examples or []):
        messages.append({"role": "user",      "content": ex["input"]})
        messages.append({"role": "assistant", "content": ex["output"]})
    messages.append({"role": "user", "content": input_text})

    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print("Utilities loaded.")


In [ ]:
# ── Load preprocessed data ────────────────────────────────────────────────────
# Run preprocessing.ipynb first to generate these files.

generic_train = pd.read_parquet(DATA_DIR / "generic_train.parquet")
domain_train  = pd.read_parquet(DATA_DIR / "domain_train.parquet")
test_df       = pd.read_parquet(DATA_DIR / "test.parquet")

# Optional: subsample test set for faster evaluation
if EVAL_SAMPLE and len(test_df) > EVAL_SAMPLE:
    test_eval = test_df.sample(EVAL_SAMPLE, random_state=SEED).reset_index(drop=True)
else:
    test_eval = test_df.reset_index(drop=True)

# Fixed few-shot examples drawn from generic training set
few_shot_examples = (
    generic_train.sample(FEW_SHOT_N, random_state=SEED)[["input", "output"]]
    .to_dict("records")
)

print(f"Generic train : {len(generic_train):,} samples")
print(f"Domain train  : {len(domain_train):,} samples")
print(f"Test (eval)   : {len(test_eval):,} / {len(test_df):,} total")
print(f"Few-shot exs  : {len(few_shot_examples)}")


In [ ]:
# ── Model loading and LoRA setup ──────────────────────────────────────────────
from unsloth import FastLanguageModel, is_bfloat16_supported


def load_model_with_lora():
    """Load the base model with Unsloth QLoRA and apply LoRA adapters."""
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,         # auto-detect (bf16 on Ampere+, fp16 on T4)
        load_in_4bit=True,  # QLoRA 4-bit quantisation
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",  # saves VRAM
        random_state=SEED,
    )
    return model, tokenizer


def free_memory(model=None):
    """Release GPU memory between experiments."""
    if model is not None:
        del model
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"VRAM used after cleanup: {used:.2f} / {total:.1f} GB")


print("Model utilities defined.")


In [ ]:
# ── Training utilities ────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig


def make_hf_dataset(df: pd.DataFrame, tokenizer) -> Dataset:
    """Convert a DataFrame to a HuggingFace Dataset in SFT chat-template format."""
    records = [
        {"text": format_chat(row["input"], row["output"], tokenizer)}
        for _, row in df.iterrows()
    ]
    return Dataset.from_list(records)


def train_on_dataset(model, tokenizer, df: pd.DataFrame, run_dir: str = "./tmp") -> object:
    """Run one phase of SFT; returns the trained model (in-memory, same object)."""
    dataset = make_hf_dataset(df, tokenizer)
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        args=SFTConfig(
            output_dir=run_dir,
            num_train_epochs=TRAIN_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            warmup_ratio=0.05,
            learning_rate=LEARNING_RATE,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            logging_steps=25,
            save_strategy="no",
            report_to="none",
            lr_scheduler_type="cosine",
            optim="adamw_8bit",
            max_seq_length=MAX_SEQ_LEN,
            dataset_text_field="text",
            seed=SEED,
        ),
    )
    trainer.train()
    return model


print("Training utilities defined.")


In [ ]:
# ── Evaluation utilities ──────────────────────────────────────────────────────

def evaluate_model(
    model, tokenizer, eval_df: pd.DataFrame,
    few_shot_examples=None, desc: str = "Evaluating",
) -> dict:
    """Switch model to inference mode, run predictions, return entity F1 metrics."""
    FastLanguageModel.for_inference(model)
    predictions  = []
    ground_truths = list(eval_df["output"])

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=desc):
        pred = generate_response(
            model, tokenizer, row["input"], few_shot_examples=few_shot_examples
        )
        predictions.append(pred)

    metrics = entity_f1(predictions, ground_truths)
    print(
        f"  P={metrics['precision']:.4f}  "
        f"R={metrics['recall']:.4f}  "
        f"F1={metrics['f1']:.4f}  "
        f"(TP={metrics['tp']} FP={metrics['fp']} FN={metrics['fn']})"
    )
    return metrics


def save_result(row: dict):
    """Append one result dict to the cumulative results CSV."""
    df_new = pd.DataFrame([row])
    if RESULTS_CSV.exists():
        df_old = pd.read_csv(RESULTS_CSV)
        df_new = pd.concat([df_old, df_new], ignore_index=True)
    df_new.to_csv(RESULTS_CSV, index=False)
    print(f"  Saved → {RESULTS_CSV}")


print("Evaluation utilities defined.")


## Baselines

Evaluate the model without any fine-tuning.

In [ ]:
# ── Baseline 1: Zero-shot ─────────────────────────────────────────────────────
print("Loading model for zero-shot baseline...")
model, tokenizer = load_model_with_lora()

print("\nRunning zero-shot evaluation...")
metrics = evaluate_model(model, tokenizer, test_eval, desc="Zero-shot")
save_result({
    "experiment": "zero_shot", "description": "Zero-shot",
    "generic_n": 0, "domain_n": 0, "sequential": False, **metrics,
})
free_memory(model)


In [ ]:
# ── Baseline 2: Few-shot (5-shot, no fine-tuning) ─────────────────────────────
print("Loading model for few-shot baseline...")
model, tokenizer = load_model_with_lora()

print(f"\nRunning {FEW_SHOT_N}-shot evaluation...")
metrics = evaluate_model(
    model, tokenizer, test_eval,
    few_shot_examples=few_shot_examples,
    desc=f"Few-shot ({FEW_SHOT_N})",
)
save_result({
    "experiment": "few_shot", "description": f"Few-shot ({FEW_SHOT_N}-shot)",
    "generic_n": 0, "domain_n": 0, "sequential": False, **metrics,
})
free_memory(model)


## Fine-tuning Experiments

Each experiment reloads the base model from scratch for a clean comparison.
Sequential configs (`Generic→Domain`) run two consecutive training phases on the **same model object** — generic first, domain second.

Results are written to the CSV after each run so interrupted sessions resume automatically.

In [ ]:
# ── Fine-tuning experiment definitions ────────────────────────────────────────
# (name, description, generic_n, domain_n, sequential)
EXPERIMENTS = [
    ("generic_500",  "Generic 0.5k",             500,   0,    False),
    ("generic_1k",   "Generic 1k",               1000,  0,    False),
    ("generic_2k",   "Generic 2k",               2000,  0,    False),
    ("generic_4k",   "Generic 4k",               4000,  0,    False),
    ("domain_500",   "Domain 0.5k",               0,   500,   False),
    ("domain_1k",    "Domain 1k",                 0,  1000,   False),
    ("domain_2k",    "Domain 2k",                 0,  2000,   False),
    ("domain_4k",    "Domain 4k",                 0,  4000,   False),
    ("seq_500_500",  "Generic→Domain 0.5k+0.5k",  500,  500,  True),
    ("seq_1k_1k",    "Generic→Domain 1k+1k",     1000, 1000,  True),
    ("seq_2k_2k",    "Generic→Domain 2k+2k",     2000, 2000,  True),
    ("seq_4k_4k",    "Generic→Domain 4k+4k",     4000, 4000,  True),
]


def run_experiment(name, description, generic_n, domain_n, sequential):
    """
    Run a single SFT experiment:
    - Reloads base model from scratch for a clean comparison.
    - Sequential configs fine-tune on generic first, then domain (same model object).
    - Results are appended to RESULTS_CSV after each experiment.
    """
    print(f"\n{'='*60}")
    print(f"  {description}")
    print(f"{'='*60}")

    model, tokenizer = load_model_with_lora()

    if sequential:
        # Phase 1 — generic
        if generic_n > 0:
            g_df = generic_train.sample(
                min(generic_n, len(generic_train)), random_state=SEED
            )
            print(f"Phase 1 — {len(g_df):,} generic samples")
            model = train_on_dataset(model, tokenizer, g_df, run_dir=f"./runs/{name}_p1")

        # Phase 2 — domain (continues from phase-1 weights)
        if domain_n > 0:
            d_df = domain_train.sample(
                min(domain_n, len(domain_train)), random_state=SEED
            )
            print(f"Phase 2 — {len(d_df):,} domain samples")
            model = train_on_dataset(model, tokenizer, d_df, run_dir=f"./runs/{name}_p2")
    else:
        if generic_n > 0:
            df = generic_train.sample(min(generic_n, len(generic_train)), random_state=SEED)
        else:
            df = domain_train.sample(min(domain_n, len(domain_train)), random_state=SEED)
        print(f"Training on {len(df):,} samples")
        model = train_on_dataset(model, tokenizer, df, run_dir=f"./runs/{name}")

    metrics = evaluate_model(model, tokenizer, test_eval, desc=f"Eval [{description}]")
    save_result({
        "experiment": name, "description": description,
        "generic_n": generic_n, "domain_n": domain_n, "sequential": sequential,
        **metrics,
    })
    free_memory(model)
    return metrics


print(f"Experiment runner defined. Total SFT configs: {len(EXPERIMENTS)}")


In [ ]:
# ── Run all fine-tuning experiments ──────────────────────────────────────────
# The loop automatically skips experiments already saved in RESULTS_CSV,
# so you can safely re-run this cell if the Colab session was interrupted.

completed = set()
if RESULTS_CSV.exists():
    completed = set(pd.read_csv(RESULTS_CSV)["experiment"].tolist())
    if completed:
        print(f"Skipping already completed: {sorted(completed)}")

for name, description, generic_n, domain_n, sequential in EXPERIMENTS:
    if name in completed:
        print(f"  [skip] {description}")
        continue
    run_experiment(name, description, generic_n, domain_n, sequential)

print("\nAll experiments complete!")


## Results

In [ ]:
# ── Display results table ─────────────────────────────────────────────────────
results = pd.read_csv(RESULTS_CSV)
cols = ["description", "generic_n", "domain_n", "sequential", "precision", "recall", "f1"]
print(results.sort_values("f1", ascending=False)[cols].to_string(index=False))


In [ ]:
# ── Plot results ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

results = pd.read_csv(RESULTS_CSV)

baselines = results[results["experiment"].isin(["zero_shot", "few_shot"])]
generic   = results[results["experiment"].str.startswith("generic_")].sort_values("generic_n")
domain    = results[results["experiment"].str.startswith("domain_")].sort_values("domain_n")
seq       = results[results["experiment"].str.startswith("seq_")].sort_values("generic_n")

sizes = [500, 1000, 2000, 4000]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# ── Left: F1 vs. training set size ───────────────────────────────────────────
for grp, label, color, marker, size_col in [
    (generic, "Generic only",      "steelblue",   "o", "generic_n"),
    (domain,  "Domain only",       "darkorange",  "s", "domain_n"),
    (seq,     "Generic→Domain",    "forestgreen", "^", "generic_n"),
]:
    f1s = grp["f1"].tolist()
    ax1.plot(sizes[:len(f1s)], f1s, marker=marker, label=label, color=color,
             linewidth=2, markersize=7)

for _, row in baselines.iterrows():
    ax1.axhline(row["f1"], linestyle="--", alpha=0.75,
                label=row["description"], color="gray")

ax1.set_xlabel("Training samples (per phase)")
ax1.set_ylabel("Entity-level F1")
ax1.set_title("F1 vs. Training Size")
ax1.set_xticks(sizes)
ax1.set_xticklabels(["0.5k", "1k", "2k", "4k"])
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(bottom=0)

# ── Right: bar chart of all experiments sorted by F1 ─────────────────────────
rs = results.sort_values("f1", ascending=True)
bar_colors = []
for _, row in rs.iterrows():
    if row["experiment"] in ["zero_shot", "few_shot"]:
        bar_colors.append("lightgray")
    elif row["experiment"].startswith("generic"):
        bar_colors.append("steelblue")
    elif row["experiment"].startswith("domain"):
        bar_colors.append("darkorange")
    else:
        bar_colors.append("forestgreen")

ax2.barh(rs["description"], rs["f1"], color=bar_colors)
ax2.set_xlabel("Entity-level F1")
ax2.set_title("All Experiment Results")
ax2.grid(True, axis="x", alpha=0.3)

# legend patches
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="lightgray",   label="Baseline"),
    Patch(facecolor="steelblue",   label="Generic only"),
    Patch(facecolor="darkorange",  label="Domain only"),
    Patch(facecolor="forestgreen", label="Generic→Domain"),
]
ax2.legend(handles=legend_elements, fontsize=9, loc="lower right")

plt.tight_layout()
plot_path = RESULTS_CSV.stem + ".png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved → {plot_path}")
